# Quick Start: GEO S-band Voice-Link Screening

This notebook shows the shortest research workflow for the toolkit: load the committed reference outputs, change one scenario, run a local Monte-Carlo screening approximation, and plot the parameter sensitivity.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from dashboard_model import (
    evaluate_voice_link,
    load_scenarios,
    load_table,
    sensitivity_rows,
)

## 1. Inspect the committed reference table

`expected_outputs/` contains the reference CSV/JSON snapshot that renders on GitHub. Running the workflow regenerates the same kinds of files under `outputs/`.

In [ ]:
baseline = pd.DataFrame(load_table("screening_baseline_comparison.csv"))
baseline[[
    "scenario_label",
    "average_snr_screening_availability",
    "single_state_lognormal_availability",
    "proposed_mixture_availability",
    "p10_ebn0_db",
]]

## 2. Pick a scenario and change proxy parameters

The values below are public-style proxy parameters. They are not proprietary handset calibration values.

In [ ]:
scenarios = load_scenarios()
scenario = next(row for row in scenarios if row["label"] == "Canyon valley")
scenario

In [ ]:
base = evaluate_voice_link(scenario, voice_rate_bps=2400.0)
stressed = evaluate_voice_link(
    scenario,
    voice_rate_bps=2400.0,
    nlos_loss_db=float(scenario["nlos_loss_db"]) + 5.0,
    added_loss_db=1.0,
)

pd.DataFrame([
    {"case": "baseline", **base},
    {"case": "+5 dB NLOS, +1 dB loss", **stressed},
])[["case", "availability", "p10_ebn0_db", "c0p01_bit_per_s_hz", "average_snr_margin_db"]]

## 3. Plot a scenario-level sensitivity ranking

In [ ]:
sens = pd.DataFrame(sensitivity_rows(scenario, voice_rate_bps=2400.0))
ax = sens.sort_values("delta_pp").plot.barh(
    x="parameter",
    y="delta_pp",
    figsize=(7.2, 3.8),
    legend=False,
    color="#2563eb",
)
ax.axvline(0, color="#444", linewidth=0.8)
ax.set_xlabel("Availability change (percentage points)")
ax.set_ylabel("")
ax.set_title("Scenario-level sensitivity around Canyon valley")
plt.tight_layout()

## 4. Regenerate the command-line outputs

For Python-only development smoke tests:

```bash
python run_all.py --skip-reference-cosim
```

For the full reference path with MATLAB, Simulink, and Communications Toolbox:

```bash
python run_all.py
```

Generated artifacts are written under `outputs/data/` and `outputs/figures/`.